In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import pandas as pd

# Daten aus dem data-Ordner laden
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))
backtesting_transaction_costs = pd.read_parquet(cfg.data_path("backtesting_costs"))
test_df = pd.read_parquet(cfg.data_path("test_data"))

### Phase 5: Evaluation & Vergleich (Der Kern der Thesis)
Beweis des Mehrwerts zur Reduktion des **Sequence of Returns Risk**.
*   **Metriken:**
    *   **Maximum Drawdown (MDD):** Wie tief war der maximale Fall? (Wichtigstes Ziel).
    *   **Sharpe Ratio / Sortino Ratio:** Risiko-adjustierte Rendite.
    *   **Calmar Ratio:** Rendite im Verhältnis zum Drawdown.
    *   **Regime-Stabilität:** Wie oft schaltet das Modell um? (LSTMs neigen zum "Overfitting" und Rauschen).

In [ ]:
import numpy as np

# --- Evaluation aus src/ ---
sys.path.insert(0, "..")
from src.backtest.evaluation import evaluate_strategies

# Daten laden
test_df = pd.read_parquet(cfg.data_path("test_data"))
backtesting_results = pd.read_parquet(cfg.data_path("backtesting_results"))

# --- DYNAMISCHE ZUORDNUNG DER SIGNALE ---
signals_to_count = pd.DataFrame(index=test_df.index)
signal_columns = [c for c in test_df.columns if c.endswith('_Signal')]

for sig_col in signal_columns:
    model_name = sig_col.rsplit('_', 1)[0]
    signals_to_count[model_name] = test_df[sig_col]

# Evaluation ausführen
evaluation_table = evaluate_strategies(backtesting_results, signals_to_count, backtesting_transaction_costs)

# Anzeige und Persistierung
print("\n--- UMFASSENDE EVALUATION (DYNAMISCH) ---")
display(evaluation_table)

evaluation_table.to_markdown(cfg.asset_path("evaluation_table"), index=True)
print(f"\nEvaluationstabelle erfolgreich unter {cfg.asset_path('evaluation_table')} persistiert.")

In [ ]:
# --- 06_MCS: Block-Bootstrap Robustness-Check ---

import pandas as pd
import os
from src.backtest.evaluation import run_monte_carlo_simulation
from src.backtest.sorr import build_sorr_scenarios
from src.backtest.plots import plot_mcs_boxplots, plot_mcs_paths, plot_mcs_quantiles

N_SIMULATIONS = cfg.evaluation.mcs.n_paths
BLOCK_SIZE = cfg.evaluation.mcs.block_length
RANDOM_SEED = cfg.evaluation.mcs.random_seed
SIM_YEARS = cfg.evaluation.mcs.sim_years
TRADING_DAYS = cfg.evaluation.mcs.trading_days_per_year
TOTAL_DAYS = SIM_YEARS * TRADING_DAYS

scenarios = build_sorr_scenarios(cfg.backtesting.sorr.scenarios)
daily_rets = backtesting_results.pct_change().dropna()

all_mc_summaries, mcs_paths_collector = run_monte_carlo_simulation(
    daily_rets=daily_rets,
    test_df=test_df,
    scenarios=scenarios,
    n_simulations=N_SIMULATIONS,
    block_size=BLOCK_SIZE,
    random_seed=RANDOM_SEED,
    sim_years=SIM_YEARS,
    trading_days_per_year=TRADING_DAYS,
)

# Boxplots
plot_mcs_boxplots(
    mcs_paths_collector, daily_rets.columns, scenarios, SIM_YEARS,
    save_path_template=os.path.join(str(cfg._base_dir / "assets"), "mcs_boxplot_{}.png"),
)

# MCS DataFrame für Pfade & Quantile
mcs_results = pd.DataFrame(mcs_paths_collector)

scenarios_list = list(vars(cfg.backtesting.sorr.scenarios).keys())
strategies = backtesting_results.columns

# Pfad-Verläufe
plot_mcs_paths(
    mcs_results, scenarios_list, strategies, cfg.color_map,
    cfg.asset_path("mcs_paths"),
    trading_days_per_year=TRADING_DAYS,
)

# Quantile
plot_mcs_quantiles(
    mcs_results, scenarios_list, strategies, TOTAL_DAYS, cfg.color_map,
    cfg.asset_path("mcs_quantiles"),
    trading_days_per_year=TRADING_DAYS,
)

# Summary
if all_mc_summaries:
    mc_multi_summary = pd.DataFrame(all_mc_summaries).set_index(['Szenario', 'Strategie'])
    mc_multi_summary.to_markdown(cfg.asset_path("mcs_summary"))
    display(mc_multi_summary)

print("Alle Monte Carlo Simulationen abgeschlossen.")

from IPython.display import Image, display
for sc_name in scenarios_list:
    boxplot_path = cfg.asset_path(f"mcs_boxplot_{sc_name.lower()}")
    display(Image(filename=boxplot_path))
display(Image(filename=cfg.asset_path("mcs_paths")))
display(Image(filename=cfg.asset_path("mcs_quantiles")))

print(mcs_results)

In [ ]:
output_path = cfg.data_path('mcs_data')

# Speichern als Parquet
mcs_results.to_parquet(output_path)

print(f"MCS-Dataframe erfolgreich unter {output_path} gespeichert.")

### Issue #13 — Erweiterte Evaluations-Metriken (Kap. 4.1–4.4)

Ulcer Index, Classification vs. NBER (Precision/Recall/F1 + Confusion Matrices + ROC/PR),
Signal-Churning & Whipsaw, Threshold-Sensitivität, Regime-Heatmap, Time-to-Recovery,
Switch-Timing relativ zu DD-Peak, MCS Depletion Rate CIs, Hypothesentests H1/H2,
Break-Even-Transaktionskosten, Entnahmeraten-Sensitivität.

In [ ]:
# --- Issue #13: Extended Evaluation ---
from pathlib import Path
from src.data.labels import load_nber_recession
from src.backtest.engine import backtest
from src.backtest.sorr import run_sorr_simulation
from src.backtest.evaluation import (
    add_ulcer_to_table,
    compute_classification_metrics, plot_confusion_matrices, plot_roc_pr_curves,
    churning_stats, threshold_sensitivity,
    time_to_recovery, switch_timing_vs_peak,
    mcs_final_capitals, depletion_rate_with_ci,
    test_h1_drawdown, test_h2_transformer, plot_mcs_violins,
    break_even_transaction_cost, plot_break_even, withdrawal_sensitivity,
    plot_regime_probability_heatmap,
)

ext = cfg.evaluation.extended
models = list(ext.f1_models)

# 1) Ulcer Index → Evaluation-Tabelle erweitern
evaluation_table = add_ulcer_to_table(backtesting_results, evaluation_table)
evaluation_table.to_markdown(cfg.asset_path("evaluation_table"), index=True)
display(evaluation_table)

# 2) Classification vs. NBER
nber = load_nber_recession(test_df.index, source=ext.nber_source)
class_tbl, cms = compute_classification_metrics(test_df, nber, models)
class_tbl.to_markdown(cfg.asset_path("classification_metrics"))
plot_confusion_matrices(cms, cfg.asset_path("confusion_matrices"))
aucs = plot_roc_pr_curves(
    test_df, nber, models, cfg.color_map,
    cfg.asset_path("roc_curves"), cfg.asset_path("pr_curves"),
)
display(class_tbl.join(aucs))

# 3) Churning + Threshold-Sensitivität
churn = churning_stats(
    test_df, models, cfg.transaction_cost_rate,
    min_phase_days=ext.whipsaw_min_phase_days,
)
churn.to_markdown(cfg.asset_path("churning_stats"))
display(churn)

for m, grid in vars(ext.threshold_grid).items():
    ts = threshold_sensitivity(
        test_df, backtest, m, list(grid),
        cfg.transaction_cost_rate, cfg.backtesting.signal_shift,
    )
    ts.to_markdown(cfg.asset_path("threshold_sensitivity").replace("{model}", m))

# 4) Regime-Wahrscheinlichkeits-Heatmap
plot_regime_probability_heatmap(
    test_df, models, cfg.asset_path("regime_probability_heatmap"),
)

# 5) Time-to-Recovery + Switch-Timing
for m in ["Buy_Hold"] + models:
    if m not in backtesting_results.columns:
        continue
    ttr = time_to_recovery(backtesting_results[m], min_dd=ext.ttr_min_dd)
    ttr.to_markdown(
        cfg.asset_path("ttr_table").replace("{model}", m), index=False,
    )

crisis_windows = {name: tuple(w) for name, w in vars(ext.crisis_windows).items()}
switch_rows = []
for m in models:
    t = switch_timing_vs_peak(test_df, backtesting_results, m, crisis_windows)
    if not t.empty:
        t.insert(0, "Modell", m)
        switch_rows.append(t.reset_index())
if switch_rows:
    pd.concat(switch_rows, ignore_index=True).to_markdown(
        cfg.asset_path("switch_timing"), index=False,
    )

# 6) MCS: Depletion-CIs + H1/H2-Hypothesentests + Violin-Plots
scenarios = list(vars(cfg.backtesting.sorr.scenarios).keys())
strategies = list(backtesting_results.columns)
finals = mcs_final_capitals(mcs_paths_collector, scenarios, strategies)

dep = depletion_rate_with_ci(finals, alpha=ext.alpha)
dep.to_markdown(cfg.asset_path("depletion_ci"))
display(dep)

regime_models = [m for m in strategies if m != "Buy_Hold"]
h1 = test_h1_drawdown(
    mcs_paths_collector,
    scenario=ext.hypothesis_scenario,
    regime_models=regime_models,
    alpha=ext.alpha,
)
h1.to_markdown(cfg.asset_path("h1_drawdown"))
display(h1)

h2 = test_h2_transformer(
    finals,
    scenario=ext.hypothesis_scenario,
    alpha=ext.alpha,
)
h2.to_markdown(cfg.asset_path("h2_transformer"))
display(h2)

plot_mcs_violins(
    finals, scenarios, strategies, cfg.color_map,
    save_path_template=str(cfg._base_dir / "assets" / "mcs_violin_{}.png"),
)

# 7) Break-Even-Transaktionskosten
be_tbl, be_curves = break_even_transaction_cost(
    test_df, backtest, backtesting_results["Buy_Hold"],
    [m for m in models if f"{m}_Signal" in test_df.columns],
    list(ext.fee_grid_bps),
    cfg.backtesting.signal_shift,
)
be_tbl.to_markdown(cfg.asset_path("break_even_table"))
plot_break_even(
    be_curves, float(backtesting_results["Buy_Hold"].iloc[-1]),
    cfg.color_map, cfg.asset_path("break_even_plot"),
)
display(be_tbl)

# 8) Entnahmeraten-Sensitivität (3.5 % / 4 % / 5 %)
wdraw = withdrawal_sensitivity(
    backtesting_results, test_df, run_sorr_simulation,
    base_scenario={
        "start": cfg.backtesting.sorr.scenarios.Standard.initial_capital,
        "fee":   cfg.backtesting.sorr.scenarios.Standard.liquidity_fee,
    },
    rates=tuple(ext.withdrawal_rates),
)
wdraw.to_markdown(cfg.asset_path("withdrawal_sensitivity"))
display(wdraw)

print("Issue #13 Evaluation abgeschlossen.")